Setup:

In [ ]:
import gc
gc.collect()

28

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.cuda.empty_cache()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3-8B', use_fast=True)
model = AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B', dtype=torch.bfloat16, device_map="auto")
model.seqlen = model.config.max_position_embeddings

/home/mrajnoha/double-block-sparse/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 291/291 [00:11<00:00, 25.40it/s] 


Tests:

In [ ]:
from llm import *

def main():
    wandb.init()
    model.eval()
    dataloader, testloader = get_c4(NSAMPLES, seed=42, seqlen=model.seqlen, tokenizer=tokenizer)

    tick = time.time()
    llama_sequential(model, dataloader)
    for n, p in model.named_parameters():
        print(n, torch.mean((p == 0).float()))
        if 'down_proj' in n:
            break
    print(time.time() - tick)

    dataloader, testloader = get_wikitext2(NSAMPLES, seed=42, seqlen=model.seqlen, tokenizer=tokenizer)
    llama_eval(model, testloader, "wikitext2", log_wandb=True)

    filepath = "./pruned/"
    model.save_pretrained(filepath)

# TODO test pipeline
# TODO fix pipeline

In [ ]:
main()

Cleanup:

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()